In [1]:

# STEP 1: FIND THE DISASTER RESPONSE CSV FILES

from pathlib import Path

# Set the path to the folder containing the raw dataset.
raw_folder = Path("../../Raw_Data/Disaster_Response_management_Data")

# Find CSV files in the folder and any subfolders.
csv_files = sorted(raw_folder.rglob("*.csv"))

# Display the filenames and their locations.
if not csv_files:
    print("No CSV files found.")
    print("Please check this folder:", raw_folder.resolve())
else:
    print("CSV files found:", len(csv_files))

    for file_path in csv_files:
        print(file_path.relative_to(raw_folder))


CSV files found: 3
disaster_response_test.csv
disaster_response_training.csv
disaster_response_validation.csv


In [2]:

# STEP 2: INSPECT THE DISASTER RESPONSE DATA SPLITS

import pandas as pd
from pathlib import Path

# Set the folder containing the original CSV files.
raw_folder = Path("../../Raw_Data/Disaster_Response_management_Data")

# Identify the three dataset splits.
file_names = [
    "disaster_response_training.csv",
    "disaster_response_validation.csv",
    "disaster_response_test.csv"
]

# Inspect each split without changing the original files.
for file_name in file_names:
    file_path = raw_folder / file_name

    print("\n" + "=" * 65)
    print("FILE:", file_name)

    # Check that the expected file exists.
    if not file_path.exists():
        print("File not found:", file_path.resolve())
        continue

    # Load the dataset.
    data = pd.read_csv(file_path, low_memory=False)

    # Display its dimensions and column names.
    print("Dataset shape:", data.shape)
    print("\nColumns:")
    print(data.columns.tolist())

    # Display data types.
    print("\nData types:")
    print(data.dtypes)

    # Display missing values.
    print("\nMissing values:")
    print(data.isna().sum().sort_values(ascending=False).head(15))

    # Display a small sample.
    print("\nFirst three rows:")
    print(data.head(3).to_string(index=False))



FILE: disaster_response_training.csv
Dataset shape: (19116, 45)

Columns:
['id', 'split', 'message', 'original', 'genre', 'related', 'PII', 'request', 'offer', 'aid_related', 'medical_help', 'medical_products', 'search_and_rescue', 'security', 'military', 'child_alone', 'water', 'food', 'shelter', 'clothing', 'money', 'missing_people', 'refugees', 'death', 'other_aid', 'infrastructure_related', 'transport', 'buildings', 'electricity', 'tools', 'hospitals', 'shops', 'aid_centers', 'other_infrastructure', 'weather_related', 'floods', 'storm', 'fire', 'earthquake', 'cold', 'other_weather', 'direct_report', 'event', 'actionable_haiti', 'date_haiti']

Data types:
id                          int64
split                         str
message                       str
original                      str
genre                         str
related                     int64
PII                         int64
request                     int64
offer                       int64
aid_related               

In [3]:

# STEP 1: FIND THE DISASTER RESPONSE CSV FILES

from pathlib import Path

# Set the path to the folder containing the raw dataset.
raw_folder = Path("../../Raw_Data/Disaster_Response_management_Data")

# Find CSV files in the folder and any subfolders.
csv_files = sorted(raw_folder.rglob("*.csv"))

# Display the filenames and their locations.
if not csv_files:
    print("No CSV files found.")
    print("Please check this folder:", raw_folder.resolve())
else:
    print("CSV files found:", len(csv_files))

    for file_path in csv_files:
        print(file_path.relative_to(raw_folder))


CSV files found: 3
disaster_response_test.csv
disaster_response_training.csv
disaster_response_validation.csv


In [4]:

# Import pandas for working with CSV data.
import pandas as pd

# Import Path to manage file and folder paths.
from pathlib import Path

# Set the path to the original training CSV file.
raw_file = Path("../../Raw_Data/Disaster_Response_management_Data/disaster_response_training.csv")

# Set the folder where the processed file will be saved.
output_folder = Path(".")

# Create the output folder if it does not exist.
output_folder.mkdir(parents=True, exist_ok=True)

# Load the original training dataset.
df = pd.read_csv(raw_file)

# Print the original dataset dimensions.
print("Original dataset shape:", df.shape)

# Count missing values before preprocessing.
missing_before = int(df.isna().sum().sum())

# Remove columns that contain no data at all.
empty_columns = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_columns)

# Clean spaces around text values without changing missing values.
text_columns = ["message", "original", "genre", "split", "event"]

for column in text_columns:
    if column in df.columns:
        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

# Convert the date column to datetime format when it exists.
# Invalid or missing dates remain missing rather than stopping the process.
if "date_haiti" in df.columns:
    df["date_haiti"] = pd.to_datetime(
        df["date_haiti"],
        errors="coerce"
    )

# Count exact duplicate rows before removing them.
duplicate_rows = int(df.duplicated().sum())

# Remove exact duplicate rows while preserving the first occurrence.
df = df.drop_duplicates().reset_index(drop=True)

# Count repeated IDs to identify possible ID-quality issues.
duplicate_ids = int(df["id"].duplicated().sum()) if "id" in df.columns else 0

# Count missing values after preprocessing.
missing_after = int(df.isna().sum().sum())

# Count missing message text.
missing_messages = int(df["message"].isna().sum()) if "message" in df.columns else 0

# Save the processed training dataset.
output_file = output_folder / "disaster_response_training_processed.csv"
df.to_csv(output_file, index=False)

# Create a report containing the preprocessing results.
report = pd.DataFrame([{
    "Original_Rows": len(pd.read_csv(raw_file)),
    "Processed_Rows": len(df),
    "Processed_Columns": len(df.columns),
    "Empty_Columns_Removed": ", ".join(empty_columns) if empty_columns else "None",
    "Duplicate_Rows_Removed": duplicate_rows,
    "Duplicate_ID_Count": duplicate_ids,
    "Missing_Messages": missing_messages,
    "Missing_Values_Before": missing_before,
    "Missing_Values_After": missing_after
}])

# Save the training dataset's preprocessing report.
report_file = output_folder / "disaster_response_training_preprocessing_report.csv"
report.to_csv(report_file, index=False)

# Display the results.
print("\n--- TRAINING DATA PREPROCESSING RESULTS ---")
print("Processed dataset shape:", df.shape)
print("Empty columns removed:", empty_columns)
print("Duplicate rows removed:", duplicate_rows)
print("Duplicate IDs:", duplicate_ids)
print("Missing messages:", missing_messages)
print("Missing values before:", missing_before)
print("Missing values after:", missing_after)
print("\nProcessed dataset saved to:", output_file.resolve())
print("Preprocessing report saved to:", report_file.resolve())

# Display the report in table form.
display(report)


C:\Users\ishit\AppData\Local\Temp\ipykernel_15620\3002704109.py:17: DtypeWarning: Columns (0: original, 1: event, 2: date_haiti) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_file)


Original dataset shape: (19116, 45)

--- TRAINING DATA PREPROCESSING RESULTS ---
Processed dataset shape: (19099, 45)
Empty columns removed: []
Duplicate rows removed: 17
Duplicate IDs: 33
Missing messages: 0
Missing values before: 43695
Missing values after: 43673

Processed dataset saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Disaster Response Messages\disaster_response_training_processed.csv
Preprocessing report saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Disaster Response Messages\disaster_response_training_preprocessing_report.csv


C:\Users\ishit\AppData\Local\Temp\ipykernel_15620\3002704109.py:69: DtypeWarning: Columns (0: original, 1: event, 2: date_haiti) have mixed types. Specify dtype option on import or set low_memory=False.
  "Original_Rows": len(pd.read_csv(raw_file)),


,Original_Rows,Processed_Rows,Processed_Columns,Empty_Columns_Removed,Duplicate_Rows_Removed,Duplicate_ID_Count,Missing_Messages,Missing_Values_Before,Missing_Values_After
0,19116,19099,45,None,17,33,0,43695,43673
